# 06 — Tools: MCP Server & Client

**Module 2 of the workshop.** The **MCP** branch of the tools diagram — a standard protocol, external tool server, reusable capability. Needs two terminals; this notebook cannot run both halves itself.


## Problem

An LLM alone can't read a file, call an API, or run code. Custom `@tool` functions solve that *inside one application* — but a tool that needs to be reusable across *different* agents/apps needs a standard protocol, not a function baked into one codebase. That's what MCP (Model Context Protocol) is for.


## Concept

Four mechanisms, one interface — the agent doesn't care which kind of tool it's calling:

```
                 TOOLS
                   │
       ┌───────────┼────────────┐
       │           │            │
     Custom      Vended        MCP
      Tools       Tools        Tools
       │           │            │
       └───────────┼────────────┘
                   │
             Agent-as-tool
```

**Tool vs MCP, the distinction that matters:**

```
Custom Tool → function inside your application
MCP         → standard protocol → external tool server → reusable capability
```

**When would you NOT use MCP specifically?** If the function is local, simple, and tightly coupled to this one application — a custom `@tool` function is simpler than standing up a protocol server for it. MCP earns its cost when the tool needs to be *reusable across different agents/apps*.


## Architecture

```
TERMINAL 1 (server)                    TERMINAL 2 (client)
┌─────────────────────┐
│ FastMCP("Calculator  │
│  Server")            │
│                      │
│  @mcp.tool           │
│  def add(x, y)       │
│                      │
│  @mcp.tool           │
│  def multiply(x, y)  │
│                      │
│ mcp.run(             │
│   transport=          │
│   "streamable-http") │
└──────────┬───────────┘
           │ listens on
           │ localhost:8000/mcp/
           │
           │◀──────────────────────────┐
           │  HTTP (MCP protocol)       │
           │                            │
           │                     ┌──────┴────────────┐
           │                     │ MCPClient           │
           │                     │ (streamablehttp_    │
           │                     │  client)             │
           │                     └──────┬──────────────┘
           │                            │ list_tools_sync()
           │                            ▼
           │                     tools = [add, multiply]
           │                            │
           │                            ▼
           │                     ┌──────────────┐
           │                     │ Agent(tools=  │
           │                     │   tools)      │
           │                     └──────┬────────┘
           │                            │ "What is 125 plus 375?"
           │                            ▼
           │◀───────────────────  calls add(125, 375) over MCP
           └──────────────────────▶ returns 500
```

The agent never knows `add`/`multiply` live in a separate process — `MCPClient.list_tools_sync()` discovers them at runtime and hands them to `Agent(tools=...)` exactly like any local tool.


## ⚠️ This notebook cannot run both halves

The server (`run_server()`) blocks forever once started (`mcp.run(...)` is a long-running listener) — it cannot share a process with the client cell that follows it. **Before running the client cell below, start the server in a separate terminal:**

```
uv run 1-mcp_calculator.py server
```

Leave that terminal running, then come back and run the client cells here.


## Step 1 — The server half (run this in a terminal, not this notebook)

`FastMCP` turns two plain Python functions into MCP-discoverable tools with one decorator each. This cell is shown for reference — do not execute it in the notebook, it will block the kernel.


In [ ]:
def run_server():
    from mcp.server import FastMCP

    mcp = FastMCP("Calculator Server")

    @mcp.tool(description="Add two numbers together")
    def add(x: int, y: int) -> int:
        return x + y

    @mcp.tool(description="Multiply two numbers together")
    def multiply(x: int, y: int) -> int:
        return x * y

    mcp.run(transport="streamable-http")

# Do NOT call run_server() here — it blocks forever.
# Run it from a terminal instead: uv run 1-mcp_calculator.py server


## Step 2 — Resolve the model (client side)

The client runs in this notebook's process, separate from the server.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model

model = get_model()
print(f"Using: {type(model).__name__}")


## Step 3 — Connect the MCP client and discover tools

`MCPClient` wraps the transport; `list_tools_sync()` asks the running server what tools it exposes — this is the runtime discovery step that makes MCP tools reusable across different clients without hardcoding a tool list.

**Make sure the server from Step 1 is already running in a separate terminal before executing this cell.**


In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.tools.mcp.mcp_client import MCPClient


def create_transport():
    return streamablehttp_client("http://localhost:8000/mcp/")


mcp_client = MCPClient(create_transport)
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Discovered tools: {[t.tool_name for t in tools]}")


## Step 4 — Build the agent with the discovered tools and run it

The agent calls `add` over MCP exactly as it would call any local `@tool` function — it never knows the tool lives in a different process.


In [ ]:
with mcp_client:
    tools = mcp_client.list_tools_sync()
    agent = Agent(model=model, tools=tools)
    response = agent("What is 125 plus 375?")
    print(response)


## Failure mode to know about

Needs two terminals — pre-start both before demoing, don't fumble it live. If the client connects before the server is up, or the server terminal was closed, every tool call fails with a connection error, not a helpful "server not running" message — check `localhost:8000/mcp/` is actually listening before debugging anything else.
